# Create schema

In [0]:
CATALOG = "dbr_dev_ua5816bd"
SCHEMA = "mialkovska_viktor594"
VOLUME = "raw_files"

RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DATASET_PATH = f"{RAW_PATH}/smart_shipment_route_monitoring"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [0]:
%pip install -q kagglehub

In [0]:
import os
import kagglehub

DATASET = "tharishreddy22/smart-shipment-logistics-and-route-event-monitoring"

if os.path.exists(DATASET_PATH) and len(os.listdir(DATASET_PATH)) > 0:
    print("Dataset already exists. Download skipped.")
else:
    kagglehub.dataset_download(
        DATASET,
        output_dir=DATASET_PATH
    )
    print("Dataset downloaded.")

print(os.listdir(DATASET_PATH))

In [0]:
DATA_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw_files/"
    "smart_shipment_route_monitoring/shipment"
)

SHIPMENTS_PATH = f"{DATA_PATH}/shipments_master.csv"
EVENTS_PATH = f"{DATA_PATH}/route_events.csv"

print(SHIPMENTS_PATH)
print(EVENTS_PATH)

display(dbutils.fs.ls(DATA_PATH))

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    BooleanType,
    DateType,
    TimestampType
)

shipments_schema = StructType([
    StructField("shipment_id", StringType(), True),
    StructField("carrier", StringType(), True),
    StructField("origin_port", StringType(), True),
    StructField("destination_port", StringType(), True),
    StructField("transport_mode", StringType(), True),
    StructField("status", StringType(), True),
    StructField("goods_category", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("created_date", DateType(), True),
    StructField("eta_date", DateType(), True),
    StructField("transit_hours", DoubleType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("volume_m3", DoubleType(), True),
    StructField("value_usd", DoubleType(), True),
    StructField("freight_cost_usd", DoubleType(), True),
    StructField("num_containers", IntegerType(), True),
    StructField("num_stops", IntegerType(), True),
    StructField("delay_hours", DoubleType(), True),
    StructField("risk_score", DoubleType(), True),
    StructField("weather_severity", DoubleType(), True),
    StructField("port_congestion", DoubleType(), True),
    StructField("temperature_c", DoubleType(), True),
    StructField("priority_level", IntegerType(), True),
    StructField("insurance_required", BooleanType(), True)
])

In [0]:
events_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_timestamp", TimestampType(), True),
    StructField("location_name", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("speed_knots", DoubleType(), True),
    StructField("heading_deg", DoubleType(), True),
    StructField("temperature_c", DoubleType(), True),
    StructField("humidity_pct", DoubleType(), True),
    StructField("shock_g", DoubleType(), True),
    StructField("delay_added_hours", DoubleType(), True),
    StructField("risk_score_delta", DoubleType(), True),
    StructField("port_wait_hours", DoubleType(), True),
    StructField("fuel_consumed_lt", DoubleType(), True),
    StructField("co2_kg", DoubleType(), True),
    StructField("sensor_type", StringType(), True),
    StructField("signal_quality", StringType(), True),
    StructField("anomaly_flag", BooleanType(), True),
    StructField("alert_sent", BooleanType(), True)
])

In [0]:
shipments_df = (
    spark.read
    .option("header", True)
    .schema(shipments_schema)
    .csv(SHIPMENTS_PATH)
)

events_df = (
    spark.read
    .option("header", True)
    .schema(events_schema)
    .csv(EVENTS_PATH)
)

In [0]:
display(events_df.limit(10))
display(shipments_df.limit(10))

In [0]:
shipments_df.printSchema()


In [0]:
events_df.printSchema()

In [0]:
SHIPMENTS_RAW = f"{CATALOG}.{SCHEMA}.shipments_raw"
EVENTS_RAW = f"{CATALOG}.{SCHEMA}.events_raw"

(
    shipments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SHIPMENTS_RAW)
)

(
    events_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(EVENTS_RAW)
)

print("Raw Delta tables created successfully.")

In [0]:
from pyspark.sql import functions as F

SHIPMENTS_RAW = f"{CATALOG}.{SCHEMA}.shipments_raw"

shipments_df = spark.table(SHIPMENTS_RAW)

locations_df = (
    shipments_df
    .select(
        F.col("origin_port").alias("location_name")
    )
    .union(
        shipments_df.select(
            F.col("destination_port").alias("location_name")
        )
    )
    .filter(F.col("location_name").isNotNull())
    .distinct()
    .orderBy("location_name")
)

print(f"Distinct locations: {locations_df.count()}")

display(locations_df)

In [0]:
def get_location_info(location_name):
    
    search_name = location_name.replace("_", " ")
    
    try:
        response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={
                "name": search_name,
                "count": 1,
                "language": "en",
                "format": "json"
            },
            timeout=20
        )
        
        response.raise_for_status()
        data = response.json()
        
        if not data.get("results"):
            return {
                "location_name": location_name,
                "resolved_name": None,
                "country": None,
                "country_code": None,
                "admin_region": None
            }
        
        result = data["results"][0]
        
        return {
            "location_name": location_name,
            "resolved_name": result.get("name"),
            "country": result.get("country"),
            "country_code": result.get("country_code"),
            "admin_region": result.get("admin1")
        }
    
    except Exception as e:
        print(f"Failed: {location_name} -> {e}")
        
        return {
            "location_name": location_name,
            "resolved_name": None,
            "country": None,
            "country_code": None,
            "admin_region": None
        }

In [0]:
get_location_info("Jebel_Ali")

In [0]:
import time

locations = [
    row["location_name"]
    for row in locations_df.collect()
]

location_results = []

for i, location in enumerate(locations, start=1):
    
    result = get_location_info(location)
    location_results.append(result)
    
    print(
        f"{i}/{len(locations)} | "
        f"{location} -> {result['country']}"
    )
    
    time.sleep(0.2)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

location_schema = StructType([
    StructField("location_name", StringType(), False),
    StructField("resolved_name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("admin_region", StringType(), True)
])

dim_location_df = spark.createDataFrame(
    location_results,
    schema=location_schema
)

display(dim_location_df)